# Coordinate systems and axes

**Worksheet · Data 110 · Fall 2026**
Rebin Muhammad

This is the worksheet for the lecture. Do it after class. Run each cell with **Shift+Enter**, from the top.

Under each exercise there is an empty cell. Write your solution in that cell, then run it. This file does not contain the answers.

## How to open this in Google Colab

1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this file.
3. Upload the two data files before you run the data cells. Click the **folder icon** on the left, then the **upload icon**, and choose `tempnormals.csv` and `US_census.csv`. The names must stay exactly those names. Colab forgets uploaded files when the session ends, so you upload them again next time.


## Setup

This cell loads the three packages we use. A package is a collection of functions written by someone else. The short names `plt`, `np`, and `pd` are the usual nicknames.


In [ ]:
import matplotlib.pyplot as plt  # pyplot draws the plots. We call it plt.
import numpy as np               # numpy does the math. We call it np.
import pandas as pd              # pandas holds tables. We call it pd.

# rcParams is matplotlib's list of default settings.
# A change here applies to every plot after this cell.
# The "spines" are the four lines that make the frame.
# False hides the top line and the right line, which we do not need.
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BLUE = "#0072B2"  # one blue, used for every chart that has a single series


## The three tables

We use the same three tables as the lecture.

- `boxoffice` is five films and the weekend gross, in million US dollars. We type this one. It is small.
- `temps` comes from `tempnormals.csv`. It is the normal temperature for each day of the year in four places. Temperature is in degrees Fahrenheit. The file includes February 29, so April 1 is day 92.
- `texas` comes from `US_census.csv`. We keep the Texas counties and compare each county's 2010 population with the typical (median) county.


### Box office

`pd.DataFrame` builds a table from columns. The last line of a cell is shown underneath the cell, so the table appears when you run this.


In [ ]:
boxoffice = pd.DataFrame({
    "title": [
        "Star Wars",
        "Jumanji",
        "Pitch Perfect 3",
        "Greatest Showman",
        "Ferdinand",
    ],
    "amount": [71.57, 36.17, 19.93, 8.81, 7.32],
})

boxoffice


`sort_values` reorders the rows. `barh` (used later) draws the first row at the **bottom**, so we put the smallest amount first.


In [ ]:
boxoffice = boxoffice.sort_values("amount")
boxoffice


### Temperatures

Upload `tempnormals.csv` if you have not yet. Then run the next cell.

`read_csv` reads a comma-separated file into a table. `head()` shows the first five rows, which is enough to see the column names.


In [ ]:
temps = pd.read_csv("tempnormals.csv")
temps.head()


`loc` picks rows. The test inside the brackets is checked once per row. `sort_values` puts January 1 first. We will reuse `houston` in several plots.


In [ ]:
houston = temps.loc[temps["location"] == "Houston"]
houston = houston.sort_values("day_of_year")
houston.head()


In [ ]:
san_diego = temps.loc[temps["location"] == "San Diego"]
san_diego = san_diego.sort_values("day_of_year")
san_diego.head()


### Texas counties

Upload `US_census.csv` if you have not yet. The file is the whole country. The next cells keep Texas and build the column the lecture plots.


In [ ]:
census = pd.read_csv("US_census.csv")
census.head()


In [ ]:
# Keep the Texas rows. copy() makes a separate table, so later edits do not change census.
texas = census.loc[census["state"] == "Texas"].copy()
texas.head()


In [ ]:
# "Harris County" becomes "Harris". regex=False means those exact characters, not a pattern.
texas["county"] = texas["name"].str.replace(" County", "", regex=False)
texas[["name", "county", "pop2010"]].head()


In [ ]:
# The median is the middle county: half the counties are smaller, half are larger.
median_pop = texas["pop2010"].median()
median_pop


In [ ]:
# popratio is "how many times the median is this county?"
# A typical county is 1. A county of 2 has twice the typical population.
texas["popratio"] = texas["pop2010"] / median_pop
texas[["county", "pop2010", "popratio"]].head()


In [ ]:
# Largest county first. reset_index(drop=True) throws away the old row numbers
# and writes new ones starting at 0. We then add 1 so the chart starts at 1.
texas = texas.sort_values("popratio", ascending=False)
texas = texas.reset_index(drop=True)
texas["index"] = texas.index + 1

print("median Texas county:", int(median_pop), "people")
print(len(texas), "counties")
texas[["index", "county", "pop2010", "popratio"]].head()


## 1. A linear change of units does not change the shape

Celsius is Fahrenheit with two steps: subtract 32, then multiply by 5/9.

$$\,^{\circ}\mathrm{C} = (\,^{\circ}\mathrm{F} - 32) \times 5/9.$$

Subtracting 32 moves the zero. Multiplying by 5/9 changes the size of one degree. Neither step bends the curve, if both axes use the same rule and one degree has the same length on each axis.

**Exercise.** In the empty cell below, write a function `fahrenheit_to_celsius(temp_f)` that does this conversion. Then print `fahrenheit_to_celsius(32)` and `fahrenheit_to_celsius(212)`. `32` should come back as `0`, and `212` should come back as `100`.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


Now draw the shape. San Diego is on the x axis and Houston is on the y axis. Each day is one point, and the points are connected in day order.

Both tables were sorted by `day_of_year`, so the first row is the same day in each city.


In [ ]:
# subplots makes the picture. fig is the whole picture. ax is the frame inside it.
# figsize is (width, height) in inches.
fig, ax = plt.subplots(figsize=(5, 5))

# plot(x, y) draws the line. color= picks the color.
ax.plot(san_diego["temperature"], houston["temperature"], color=BLUE)

# set_aspect("equal") makes one degree the same length on x and on y.
# Without it, the window can stretch the curve.
ax.set_aspect("equal")

ax.set_xlabel("San Diego (F)")  # set_xlabel is the name under the x axis
ax.set_ylabel("Houston (F)")    # set_ylabel is the name beside the y axis
ax.set_title("Fahrenheit")      # set_title is the line above the plot

plt.show()  # show() draws the picture


**Exercise.** Make the same plot in Celsius. The function works on a whole column, not only on one number. Keep `set_aspect("equal")`. The curve should have the same shape.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


## 2. When the units differ, the aspect ratio is your choice

Time and temperature are not the same kind of measurement. Nothing in the data says how wide the plot should be. A steeper curve can just be a narrower frame.

The x values are day numbers. Five of those numbers are turned into month names. April 1 is day 92, July 1 is day 183, and October 1 is day 275, because the file includes February 29.


In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.4))  # a narrow picture
ax.plot(houston["day_of_year"], houston["temperature"], color=BLUE)

# set_ylim(low, high) decides where the y axis starts and stops.
ax.set_ylim(50, 90)
ax.set_ylabel("temperature (F)")

# set_xlim is the same idea for x.
# set_xticks chooses where the tick marks go.
# set_xticklabels chooses the words written at those marks. The order must match.
ax.set_xlim(1, 366)
ax.set_xticks([1, 92, 183, 275, 366])
ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct", "Jan"])
ax.set_xlabel("month")
ax.set_title("Narrow frame")
plt.show()


**Exercise.** In the next cell, change `figsize` from `(4.2, 3.4)` to `(8, 3)`, and change the title to `"Wide frame"`. The data and the y limit stay the same. Look at the slope before you decide that summer rises faster.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


## 3. What you can set on an axis

Start with the bars. `barh` draws horizontal bars. The first argument is the names, the second is the lengths.

The axis title still says `amount` and `title`. Those are column names, not a description of the numbers.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(boxoffice["title"], boxoffice["amount"], color=BLUE)
ax.set_xlabel("amount")
ax.set_ylabel("title")
plt.show()


**Exercise.** The axis title says what the number is, including the unit. Change the x-axis title to `weekend gross (million USD)`. Change the y-axis title to `""`, an empty string, so the film names stand alone.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


**Exercise.** `set_xlim(low, high)` sets where the axis starts and stops. A bar encodes length, so the axis starts at 0. End it at 80, a little past the longest bar.

In the book this is `limits` inside `scale_x_continuous`. Here it is `set_xlim`.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


**Exercise.** Draw the same bars again, but end the x axis at 65 instead of 80.

Star Wars grossed 71.57 million, which is past 65. The row is still in `boxoffice`. Matplotlib cuts the bar off at the edge of the axis. The table was not filtered.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


**Exercise.** Now remove the row instead of cutting the bar. `loc` keeps the films whose `amount` is at most 65. Plot `kept`, not `boxoffice`. Star Wars should be gone.

Do not use a cutoff like 65 on a bar chart you mean to publish. This is only so you can see the difference. Clipping the axis and removing a row are different operations.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


**Exercise.** Ticks are the positions you ask the reader to use. Put ticks at 0, 25, 50, and 75. Then replace the tick text with `0`, `$25M`, `$50M`, and `$75M`, in that same order. The title can drop the unit, because the labels now carry it.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


**Exercise.** Leave a little room past the last tick so the longest bar is not pressed against the frame. Keep the ticks at 0, 25, 50, and 75. Change the limit from 80 to 85.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


## 4. A bar includes zero. A line does not have to.

The length of a bar is the amount. If the axis does not start at zero, a short bar can look long.

A line shows change. Houston runs from about 54°F to about 84°F. Keeping 0°F on that axis fills the panel with empty space, and the summer rise looks flatter. Zero belongs on the temperature axis only when "none" is part of the comparison.

The next cell is Houston with the axis starting at 50°F. The month labels are the same four lines as before.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(houston["day_of_year"], houston["temperature"], color=BLUE)
ax.set_ylim(50, 90)
ax.set_ylabel("temperature (F)")
ax.set_xlim(1, 366)
ax.set_xticks([1, 92, 183, 275, 366])
ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct", "Jan"])
ax.set_xlabel("month")
ax.set_title("Axis from 50 F")
plt.show()


**Exercise.** Draw the same line again, and set the y limit from 0 to 140. Same data. A different window.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


## 5. A log scale measures ratios

On a linear scale, equal distance means equal difference: from 1 to 2 is the same gap as from 101 to 102. On a log scale, equal distance means equal ratio: from 1 to 10 is the same gap as from 10 to 100.

The column `popratio` does not change. Only the positions do.

There is no tick at zero on a log scale. The log of 0 is not a real number, and a population ratio cannot be negative.

`scatter` draws one dot per row, with no line connecting them. `s=8` is the size of each dot.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(texas["index"], texas["popratio"], s=8, color=BLUE)
ax.set_xlabel("counties, most to least populous")
ax.set_ylabel("population / median")
ax.set_title("Linear")
plt.show()


**Exercise.** Draw the county scatter again. Add `ax.set_yscale("log")`. That one line switches the y axis from equal differences to equal ratios.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


Titles, tick positions, and tick labels work on a log scale the same way they worked on the bar chart. A tick at 0 does not. The breaks below start at 0.01, which means one hundredth of the median county.

**Exercise.** On the log plot, set the y ticks to `0.01`, `1`, and `100`, and set the tick labels to those same three strings. `set_yticks` and `set_yticklabels` are the y-axis versions of the x functions from the bar chart.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


**Exercise.** Harris and Dallas sit next to each other at the top of that cloud, so the gap looks small. Divide the populations, not the positions on the plot. You should get a number near 1.7.

`iloc[0]` takes the first value in a column. After the filter there is only one row, so that value is the population.


### Your solution

Print Harris divided by Dallas. You should get a number near 1.7. Then plot the eight largest counties on a log scale, with the county names on the x axis.

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


## 6. Polar coordinates

A polar point is not (right, up). The first number is an angle. The second is the distance from the center. The pair is often written $(\theta, r)$.

You have to know where angle 0 is, and which way the angle grows.

- In a math class, 0° points to the right and the angle increases counterclockwise. Matplotlib does this unless you change it.
- In this lecture, matching the book, 0° points up and the angle increases clockwise, like the hands of a clock.

Both plots below use the same numbers: 90° and a distance of 3. They do not land in the same place.

`np.pi` is the number $\pi$, about 3.1416. A half turn is $\pi$ radians, which is 180°. A quarter turn is $\pi/2$, which is 90°.


In [ ]:
# figure() starts an empty picture.
# add_subplot(projection="polar") adds round axes.
# On round axes, scatter's first list is the angle and the second list is the distance.
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="polar")

ax.scatter([np.pi / 2], [3], s=60, color=BLUE)
ax.set_ylim(0, 4)  # distance axis, from the center out to 4
ax.set_title("Math class: (90 degrees, 3) is up")
plt.show()


**Exercise.** Plot the same point again: 90 degrees, distance 3. Put angle 0 at the top, and make the angle grow clockwise. The numbers in `scatter` do not change. The point should move to the right.

`set_theta_zero_location("N")` puts angle 0 at the top. N means north. `set_theta_direction(-1)` makes the angle grow clockwise. The default direction is `1`, counterclockwise.


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


A year is a cycle. A Cartesian axis has to cut it, so January sits at both ends. Polar coordinates close the year: day of year becomes the angle, and temperature becomes the distance from the center.

First, the ordinary plot. Each city is one `plot` call. `label=` stores the name for the legend. `legend()` prints those names. `frameon=False` leaves off the box around the legend.


In [ ]:
death_valley = temps.loc[temps["location"] == "Death Valley"].sort_values("day_of_year")
chicago = temps.loc[temps["location"] == "Chicago"].sort_values("day_of_year")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(death_valley["day_of_year"], death_valley["temperature"], color="#E69F00", label="Death Valley")
ax.plot(houston["day_of_year"], houston["temperature"], color="#56B4E9", label="Houston")
ax.plot(san_diego["day_of_year"], san_diego["temperature"], color="#009E73", label="San Diego")
ax.plot(chicago["day_of_year"], chicago["temperature"], color="#CC79A7", label="Chicago")

ax.set_xlim(1, 366)
ax.set_xticks([1, 92, 183, 275, 366])
ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct", "Jan"])
ax.set_xlabel("month")
ax.set_ylabel("temperature (F)")
ax.legend(frameon=False)
ax.set_title("Cartesian. January is cut apart.")
plt.show()


Matplotlib wants the angle in radians. One full turn is $2\pi$.

Day 1 should be angle 0. The last day should be almost one full turn. `(day - 1) / 365` is the fraction of the year. Multiply by `2 * np.pi` to get the angle.

The next cell prints that angle for Houston's first few days, so you can see the numbers before they become a plot.


In [ ]:
houston_angle = (houston["day_of_year"] - 1) / 365 * 2 * np.pi
houston[["day_of_year"]].assign(angle=houston_angle).head()


The next cell draws the four cities on a polar plot and does not set a distance limit. Matplotlib then starts the center at the coldest day in the data, about 25°F, instead of at 0°F. Chicago's winter falls into the middle and looks like almost nothing.

Because distance is measured from the center, the center has to mean none.

**Exercise.** In the empty cell after that plot, draw the same polar chart with the distance axis running from 0 to 105. Chicago's winter should become a ring around the center, not a collapse into it.


In [ ]:
death_valley_angle = (death_valley["day_of_year"] - 1) / 365 * 2 * np.pi
san_diego_angle = (san_diego["day_of_year"] - 1) / 365 * 2 * np.pi
chicago_angle = (chicago["day_of_year"] - 1) / 365 * 2 * np.pi

fig = plt.figure(figsize=(6.2, 5.4))
ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N")   # 0 at the top, so January is at the top
ax.set_theta_direction(-1)        # clockwise, like a clock

ax.plot(death_valley_angle, death_valley["temperature"], color="#E69F00", label="Death Valley")
ax.plot(houston_angle, houston["temperature"], color="#56B4E9", label="Houston")
ax.plot(san_diego_angle, san_diego["temperature"], color="#009E73", label="San Diego")
ax.plot(chicago_angle, chicago["temperature"], color="#CC79A7", label="Chicago")

ax.legend(frameon=False, bbox_to_anchor=(1.25, 1.05))
plt.show()


### Your solution

Write your code in the next cell, then run it. This notebook does not include the answer.


In [ ]:
# Write your solution here.


## What to keep

- **Cartesian.** A point is $(x, y)$. A linear change of units, applied to both axes and drawn at equal scale, does not change the shape.
- **Zero.** A bar includes zero, because length is the amount. A line includes zero only when zero belongs in the comparison.
- **Aspect ratio.** If the axes measure different things, you chose the shape of the plot.
- **Log scale.** Equal distance means "times," not "plus." A short step can still be a large ratio. There is no zero.
- **Polar.** The first number is an angle, measured from a starting direction you have to know. Use it when one variable is a cycle, and set the center to a number that means none.

The argument follows Claus O. Wilke, *Fundamentals of Data Visualization*, Chapter 3. This worksheet is the Python version for Data 110, Rebin Muhammad, Fall 2026.
